# AVCORP — Stage 5: Reasoning Trace Generation (ReAct + ReCon)

This notebook generates **reasoning gold** annotations for all 1,000 rows of `Deception-Dataset.csv` using two prompting frameworks:

- **ReAct** (Reason-Then-Act; Yao et al. 2023): Interleaved Thought → Act → Observation cycles that alternate internal reasoning with concrete assessment steps.
- **ReCon** (Recursive Contemplation; Wang et al. 2023): Two-pass contemplation — Formulation (first-order perspective) followed by Refinement (second-order, modeling adversary influence strategy).

Both frameworks produce the same structured JSON output schema (abduction, suspicion, beliefs, statement, deduction).

**Processing strategy:**
- Games are processed sequentially (G001 → G250).
- Within each game, rounds are generated in order (R1 → R2 → ... → Rn).
- Each round receives **cumulative public history** and **cumulative discussion log** (all prior rounds concatenated).
- Each round also receives all **prior-round reasoning traces** from the same framework chain.
- The first 25 seed annotations (G001–G025 R1) are **regenerated** (known annotation errors exist).

**Output:** Two separate CSVs saved to `Datasets/reasoning/`:
- `reasoning_react.csv` — 9 original columns + `reasoning_gold_react`
- `reasoning_recon.csv` — 9 original columns + `reasoning_gold_recon`

**Model:** `gpt-5.4` via OpenAI API


## Section 1: Imports and Configuration

In [1]:
import os
import json
import time
import re
import ast
import pandas as pd
from pathlib import Path
from openai import OpenAI
import backoff
from dotenv import load_dotenv
from tqdm.notebook import tqdm
from datetime import datetime

# -- Environment ---------------------------------------------------------------
load_dotenv(override=True)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# -- Constants -----------------------------------------------------------------
MODEL               = "gpt-5.4"
TEMPERATURE         = 0.5
MAX_TOKENS          = 4096
MAX_API_RETRIES     = 6
MAX_BACKOFF_SECONDS = 300
HUMAN_FLAG_TAG      = "HUMAN_REVIEW_REQUIRED"

DATASET_PATH        = Path("Deception-Dataset.csv")
OUTPUT_DIR          = Path("Datasets/reasoning")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REACT_CSV_PATH      = OUTPUT_DIR / "reasoning_react.csv"
RECON_CSV_PATH      = OUTPUT_DIR / "reasoning_recon.csv"

# The 9 original columns preserved in output CSVs (matches dialogue-gen pipeline)
PRESERVE_COLS = [
    "game_id", "round_id", "role_id", "llm_alignment",
    "player_roles", "public_history", "prior_summary_gold",
    "discussion_log", "matrix_tactic_scale"
]

# Protagonist rotates every 50 games (always Good player)
# G001-G050 -> P1, G051-G100 -> P2, G101-G150 -> P3, G151-G200 -> P4, G201-G250 -> P5
def get_protagonist(game_id: str) -> str:
    """Return the protagonist player ID (always Good) for a given game_id."""
    num = int(game_id[1:])
    if num <= 50:
        return "P1"
    if num <= 100:
        return "P2"
    if num <= 150:
        return "P3"
    if num <= 200:
        return "P4"
    return "P5"


print(f"Model: {MODEL}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Temperature: {TEMPERATURE}")
print(f"Max output tokens: {MAX_TOKENS}")
print(f"API retries: {MAX_API_RETRIES}")
print(
    f"Protagonist examples: "
    f"G001->{get_protagonist('G001')}, "
    f"G075->{get_protagonist('G075')}, "
    f"G200->{get_protagonist('G200')}"
)


Model: gpt-5.4
Output directory: D:\Projects\Avalon-deception\Datasets\reasoning
Temperature: 0.5
Max output tokens: 4096
API retries: 6
Protagonist examples: G001->P1, G075->P2, G200->P4


## Section 2: Load Dataset

In [3]:
df = pd.read_csv(DATASET_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Unique games: {df['game_id'].nunique()}")
print(f"Rounds per game distribution:\n{df.groupby('game_id')['round_id'].count().value_counts().sort_index()}")


def get_game_rounds(game_id: str) -> pd.DataFrame:
    """Return all rows for a given game_id, sorted by round_id."""
    rounds = df[df["game_id"] == game_id].copy()
    rounds = rounds.sort_values("round_id").reset_index(drop=True)
    return rounds


# Confirm reasoning_gold status
has_gold = df['reasoning_gold'].notna() & (df['reasoning_gold'].astype(str).str.strip() != '')
print(f"\nRows with existing reasoning_gold: {has_gold.sum()}")
print(f"Games with existing reasoning_gold: {df[has_gold]['game_id'].unique()[:5]} ...")
print("get_game_rounds() defined.")


Dataset shape: (1000, 11)
Columns: ['game_id', 'round_id', 'role_id', 'llm_alignment', 'player_roles', 'public_history', 'prior_summary_gold', 'discussion_log', 'matrix_tactic_scale', 'reasoning_gold', 'Overall_with_formula']
Unique games: 250
Rounds per game distribution:
round_id
3    100
4     50
5    100
Name: count, dtype: int64

Rows with existing reasoning_gold: 0
Games with existing reasoning_gold: [] ...
get_game_rounds() defined.


## Section 3: Cumulative Context Builder

`public_history` and `discussion_log` in the CSV are **per-round only** (confirmed).  
For Round N we concatenate all rows from Round 1 through Round N to form the cumulative context passed to the model.


In [4]:
# SECTION 3: Build cumulative context per game + format prior traces

def build_cumulative_history_and_discussion(game_rounds: pd.DataFrame):
    """
    For each round row, build cumulative strings:
      - cumulative_public_history: all public_history from R1..current
      - cumulative_discussion_log: all discussion_log from R1..current
    Returns a dict keyed by normalized round_id string (e.g., "1", "2", ...).
    """
    by_round = game_rounds.sort_values("round_id").copy()
    cum_hist = []
    cum_disc = []
    out = {}

    for _, row in by_round.iterrows():
        r = str(row["round_id"]).strip()
        r_label = r if r.upper().startswith("R") else f"R{r}"
        hist = str(row.get("public_history", "")).strip()
        disc = str(row.get("discussion_log", "")).strip()

        if hist:
            cum_hist.append(f"[{r_label}] {hist}")
        if disc:
            cum_disc.append(f"[{r_label}] {disc}")

        out[r] = {
            "cumulative_public_history": "\n".join(cum_hist),
            "cumulative_discussion_log": "\n".join(cum_disc),
            "player_roles": str(row.get("player_roles", "")),
        }

    return out


def format_prior_traces(prior_traces: list, framework_name: str) -> str:
    """
    prior_traces is a list of (round_id, trace_dict) for completed previous rounds.
    Returns a compact text block to include in prompt context.

    Round 1 has no prior traces and returns an empty string.
    """
    if not prior_traces:
        return ""

    parts = [
        "[PRIOR REASONING TRACES] Use these to maintain cross-round consistency."
    ]

    for r, t in prior_traces:
        r_txt = str(r)
        r_label = r_txt if r_txt.upper().startswith("R") else f"R{r_txt}"
        parts.append(f"\nRound {r_label} ({framework_name}):")
        try:
            parts.append(json.dumps(t, ensure_ascii=False))
        except Exception:
            parts.append(str(t))

    return "\n".join(parts)


# Quick sanity check on one game
sample_game = sorted(df["game_id"].unique())[0]
g = get_game_rounds(sample_game)
ctx_map = build_cumulative_history_and_discussion(g)
round_keys = sorted(ctx_map.keys(), key=lambda x: int(str(x).replace("R", "")))
print("Sample game:", sample_game)
print("Rounds:", round_keys)
print("First round cumulative history length:", len(ctx_map[round_keys[0]]["cumulative_public_history"]))
print("Last round cumulative discussion length:", len(ctx_map[round_keys[-1]]["cumulative_discussion_log"]))


Sample game: G001
Rounds: ['1', '2', '3']
First round cumulative history length: 91
Last round cumulative discussion length: 2839


## Section 4: Prompt Engineering — ReAct and ReCon Frameworks

### Design Notes

**ReAct (Yao et al. 2023 adapted for Avalon):**  
The core idea is to *interleave* explicit internal reasoning (Thought) with concrete assessment actions (Act) and evidence integration (Observation). The seven-step loop maps as:  
- THOUGHT 1 → Abductive hypothesis formation (for each player)  
- ACT 1 → Suspicion level assignment (based on T1)  
- OBSERVATION 1 → Synthesis of cumulative evidence patterns  
- THOUGHT 2 → First-order belief attribution (what each player believes)  
- ACT 2 → Second-order influence attribution (only for Strong/Extreme players)  
- OBSERVATION 2 → Final integration before generating JSON  
- THOUGHT 3 (SYNTHESIS) → Final synthesis for beliefs, public statement, and deduction  

**ReCon (Wang et al. 2023 adapted for Avalon):**  
Two-pass *recursive contemplation*:  
1. **Formulation Contemplation** — First-order perspective transitions: reason about each player from your own vantage point.  
2. **Refinement Contemplation** — Second-order perspective transitions: model how high-suspicion players are strategically shaping your beliefs. Revise analysis accordingly.  

**Critical Protagonist Constraint (encoded in system prompt):**  
The protagonist is *always* Good and *always* submits PASS. If the protagonist was on a FAILED quest, all other team members on that quest *definitively* submitted FAIL → they are Evil with certainty (Extreme suspicion, Evil abduction choice).


In [5]:
# Minimal system prompt: role assignment + reasoning mode (formatted once per game)
SYSTEM_PROMPT = (
    "You are {protagonist_id}, the Investigator \u2014 a Good player in a modified 5-player Avalon game. "
    "Reason carefully and produce only valid JSON."
)

# Static game context: prepended to every user prompt (identical across all rounds and games)
STATIC_CONTEXT = """\
AVALON OVERVIEW
Avalon is a social deduction game of hidden roles. This dataset uses a modified 5-player variant \
where the Investigator (you) has a special win condition: correctly identifying both Evil players \
wins the game regardless of quest outcomes.

GAME SETUP
5 players total: 3 Good, 2 Evil. Evil players know each other's identities; Good players know only \
their own alignment. Good wins by passing 3 quests; Evil wins by failing 3. \
Team sizes: R1=2, R2=3, R3=2, R4=3, R5=3.

ROUND SEQUENCE
(1) Team Selection: a rotating leader proposes a team. 
(2) Voting: all players vote yes/no; majority wins; in this dataset all proposed teams are approved. 
(3) Quest Execution: selected members anonymously submit PASS or FAIL; one FAIL card fails the quest. 
(4) Discussion: all players discuss the outcome, voice suspicions, and update coalitions. 
Good players must always submit PASS. Evil players may submit PASS to maintain cover or FAIL to sabotage.

YOUR ROLE
You are always Good. You know your own alignment with certainty.
Special win condition: regardless of quest outcomes, you win if you correctly identify both Evil \
players before the game ends.

CRITICAL SELF-KNOWLEDGE
- You always submitted PASS on every quest you participated in; never FAIL.
- Definitive Deduction Rule: if you were on a 2-person team and the quest FAILED, the other team \
member is definitively Evil. Assign that player Extreme suspicion, choice=Evil, and include them \
in deduction. This is a logical certainty from your own PASS submission, not an inference.
- If you were on a 3-person failed team, at least one of the other two submitted FAIL, but you cannot \
determine which. Assign Moderate-to-Extreme suspicion based on the game context and player statements \
and hedge your deduction.
- If you were not on the team, infer alignments from voting patterns, quest outcomes, and discussion \
behavior only. Give more emphasis on the discussion behaviour, statements of players, and their \
interactions to deduce alignments.

INTERNAL REASONING FRAMEWORK (apply all six steps every round)
Step 1: Abductive Hypothesis: For each non-protagonist player, form one Good-aligned explanation \
(good_expl) and one Evil-aligned explanation (evil_expl) of their behavior. Choose the more plausible \
explanation (choice) based on mission outcomes, voting behavior, and statements.
Step 2: Suspicion Level: Assign one level from Minimal, Mild, Moderate, Strong, or Extreme. \
Provide a brief rationale for your assignment.
Step 3: First-Order Belief Attribution: For each player, infer what they likely believe about \
the game state based on observable evidence.
Step 4: Second-Order Influence Attribution: For players with Strong or Extreme suspicion only, \
infer what they want other players to believe. Are they earning trust, deflecting blame, or \
responding to accusations?
Step 5: Discussion Statement: Produce a 1-2 sentence in-character dialogue that helps the Good \
team without exposing your private reasoning or role.
Step 6: Deduction: List 1 to 3 players you believe are most likely Evil. Players with Strong or \
Extreme suspicion must be included. Generally, 2 is preferred. If suspicion is genuinely tied across \
3 candidates, include all three; if only one player clearly stands out, include just that one. At the game ending round, you **MUST** include both Evil players in your deduction to win.
"""

# Full schema reminder: appended to the end of every round prompt
OUTPUT_SCHEMA_REMINDER = """\
Player ordering rule: In abduction, suspicion, and beliefs level-1, list players in their speaking \
order from the discussion log (order of first appearance). For beliefs level-2, follow the same \
speaking order but include only players with Strong or Extreme suspicion.
Return ONLY the following JSON with the exact field names; no markdown fences, no prose before or after:
{
  "abduction": [
    {"player": "P#", "good_expl": "brief Good-aligned explanation", "evil_expl": "brief Evil-aligned explanation", "choice": "Good|Evil"},
    {"player": "P#", "good_expl": "brief Good-aligned explanation", "evil_expl": "brief Evil-aligned explanation", "choice": "Good|Evil"},
    {"player": "P#", "good_expl": "brief Good-aligned explanation", "evil_expl": "brief Evil-aligned explanation", "choice": "Good|Evil"},
    {"player": "P#", "good_expl": "brief Good-aligned explanation", "evil_expl": "brief Evil-aligned explanation", "choice": "Good|Evil"}
  ],
  "suspicion": [
    {"player": "P#", "level": "Minimal|Mild|Moderate|Strong|Extreme", "reason": "brief rationale"},
    {"player": "P#", "level": "Minimal|Mild|Moderate|Strong|Extreme", "reason": "brief rationale"},
    {"player": "P#", "level": "Minimal|Mild|Moderate|Strong|Extreme", "reason": "brief rationale"},
    {"player": "P#", "level": "Minimal|Mild|Moderate|Strong|Extreme", "reason": "brief rationale"}
  ],
  "depth": 1,
  "beliefs": [
    {"level": 1, "player": "P#", "content": "what this player likely believes about game state"},
    {"level": 1, "player": "P#", "content": "what this player likely believes about game state"},
    {"level": 1, "player": "P#", "content": "what this player likely believes about game state"},
    {"level": 1, "player": "P#", "content": "what this player likely believes about game state"},
    {"level": 2, "player": "P#", "content": "what this player wants others to believe (only Strong/Extreme players)"}
  ],
  "statement": "your 1-2 sentence in-character dialogue",
  "deduction": ["P#", "P#"]
}

Schema constraints:
- abduction: exactly 4 entries, in speaking order.
- suspicion: exactly 4 entries, same order as abduction. Choose between the five levels: Minimal, Mild, Moderate, Strong, Extreme. Provide a brief rationale for each.
- depth: set to 1 unless any player has Strong or Extreme suspicion; then 2.
- beliefs level-1: exactly 4 entries, same order as abduction.
- beliefs level-2: 0 or more entries, only for Strong/Extreme players, in speaking order.
- deduction: 1 to 3 player IDs. Players with Strong or Extreme suspicion must be included. \
Generally, 2 is preferred. However, if suspicion is genuinely tied across 3 candidates, include \
all three or if only one player stands out, include just that one.
"""


print("System prompt and static context defined.")
print(f"SYSTEM_PROMPT: {len(SYSTEM_PROMPT.split())} words")
print(f"STATIC_CONTEXT: ~{len(STATIC_CONTEXT.split())} words")
print(f"OUTPUT_SCHEMA_REMINDER: ~{len(OUTPUT_SCHEMA_REMINDER.split())} words")


System prompt and static context defined.
SYSTEM_PROMPT: 22 words
STATIC_CONTEXT: ~523 words
OUTPUT_SCHEMA_REMINDER: ~340 words


In [7]:
# SECTION 4: Prompt builders (ReAct and ReCon)


def _get_speaking_order(discussion_log: str, protagonist: str) -> list:
    """Extract non-protagonist speaking order by first appearance in the discussion log."""
    pattern = re.compile(r'\b(P[1-5])\s*:', re.IGNORECASE)
    seen = []
    for match in pattern.finditer(discussion_log or ""):
        p = match.group(1).upper()
        if p != protagonist and p not in seen:
            seen.append(p)
    # Append any players not found in the log (maintains 4-entry guarantee)
    for p in ["P1", "P2", "P3", "P4", "P5"]:
        if p != protagonist and p not in seen:
            seen.append(p)
    return seen


def build_react_prompt(
    game_id: str,
    round_id: str,
    protagonist: str,
    row_context: dict,
    prior_react_text: str,
    speaking_order: list = None,
    is_final_round: bool = False,
) -> str:
    """
    ReAct-style user prompt (Yao et al., ICLR 2023), adapted for modified Avalon.
    Interleaved Thought -> Action -> Observation steps ground each reasoning step in evidence.
    Anti-loop policy: if Observation N only restates Thought N with no new evidence, select a
    genuinely different clue before proceeding.
    Sparse-thought policy: target the single most diagnostic clue per step rather than
    reviewing all evidence exhaustively.
    Thought 3 synthesizes beliefs, statement, and final deduction from the two prior T/A/O steps.
    """
    cumulative_history = row_context.get("cumulative_public_history", "")
    cumulative_discussion = row_context.get("cumulative_discussion_log", "")
    round_label = round_id if str(round_id).upper().startswith("R") else f"R{round_id}"
    order_str = ", ".join(speaking_order) if speaking_order else "per discussion log"

    lines = [
        f"CURRENT GAME CONTEXT",
        f"Game: {game_id} | Round: {round_label} | Protagonist: {protagonist}",
        f"Player speaking order this round: {order_str}",
        "",
        "**CUMULATIVE PUBLIC HISTORY**",
        cumulative_history,
        "",
        "**CUMULATIVE DISCUSSION LOG**",
        cumulative_discussion,
    ]

    if prior_react_text:
        lines += ["", "**PRIOR ROUND REASONING (ReAct chain)**", prior_react_text]

    lines += [
        "",
        "Reasoning (ReAct -> adapted for modified Avalon):",
        "",
        "Thought 1: Identify the single most diagnostic unresolved question about player alignments "
        "after this round. Focus on quest outcomes, team composition, and player statements relative "
        "to your own participation.",
        "",
        "Action 1: Select one concrete, named piece of evidence from the cumulative public history "
        'above (e.g., "R2 quest failed with team P2, P4, P5"). State it explicitly.',
        "",
        "Observation 1: State precisely what that evidence implies about one or more players' "
        "alignments. Apply the Definitive Deduction Rule if applicable.",
        "",
        "Thought 2: Based on Observation 1, state your updated hypothesis. Which player is your "
        "top suspect and on what basis? (Anti-loop: if Thought 2 would only restate Thought 1 with "
        "no new evidence, examine a genuinely different clue before continuing.)",
        "",
        "Action 2: Cross-check your hypothesis against one specific statement or voting behavior "
        "from the cumulative discussion log above. Cite the player and the content explicitly.",
        "",
        "Observation 2: Does the cross-check confirm, revise, or reinforce your Thought 2 "
        "hypothesis? State your final confidence level.",
        "",
        "Thought 3 (Synthesis): Based on your two-step analysis above, synthesize your final "
        "assessment. Which players warrant second-order belief analysis (Strong/Extreme suspicion)? "
        "What would you say to the group that is grounded only in observable evidence? State your final deduction of 1 to 3 Evil players.",
    ]

    if is_final_round:
        lines += [
            "",
            "FINAL ROUND: This is your definitive identification. "
            "You must name exactly 2 Evil players in deduction; no more, no fewer.",
        ]

    lines += [
        "",
        f"Now produce the final JSON, maintaining speaking order ({order_str}) "
        "in abduction, suspicion, and beliefs:",
        OUTPUT_SCHEMA_REMINDER,
    ]

    round_block = "\n".join(lines)
    return STATIC_CONTEXT + "\n\n---\n\n" + round_block


def build_recon_prompt(
    game_id: str,
    round_id: str,
    protagonist: str,
    row_context: dict,
    prior_recon_text: str,
    speaking_order: list = None,
    is_final_round: bool = False,
) -> str:
    """
    ReCon-style user prompt (Wang et al., 2024 -- "Avalon's Game of Thoughts"), adapted for
    modified Avalon. Recursive Contemplation: Formulation pass (first-order perspective) ->
    Refinement pass (second-order) -> JSON output.
    Refinement discipline uses three checks: Evidence Sufficiency, Cross-Round Consistency,
    and Public Statement Alignment, all grounded in observable evidence.
    """
    cumulative_history = row_context.get("cumulative_public_history", "")
    cumulative_discussion = row_context.get("cumulative_discussion_log", "")
    round_label = round_id if str(round_id).upper().startswith("R") else f"R{round_id}"
    order_str = ", ".join(speaking_order) if speaking_order else "per discussion log"

    lines = [
        f"CURRENT GAME CONTEXT",
        f"Game: {game_id} | Round: {round_label} | Protagonist: {protagonist}",
        f"Player speaking order this round: {order_str}",
        "",
        "**CUMULATIVE PUBLIC HISTORY**",
        cumulative_history,
        "",
        "**CUMULATIVE DISCUSSION LOG**",
        cumulative_discussion,
    ]

    if prior_recon_text:
        lines += ["", "**PRIOR ROUND REASONING (ReCon chain)**", prior_recon_text]

    lines += [
        "",
        "Reasoning (ReCon -> adapted for modified Avalon):",
        "",
        "Formulation contemplation (first-order perspective transition):",
        f"For each non-protagonist player in speaking order ({order_str}), infer their probable "
        "alignment based on quest participation, voting record, and speech in the discussion log. "
        "Adopt each player's perspective: given what they know, would an Evil player in their "
        "position behave differently from a Good player here? Form an initial role assessment and "
        "suspicion level for each player, grounded only in observable evidence.",
        "",
        "Refinement contemplation (second-order perspective transition):",
        "Revisit your initial formulation from the perspectives of other players. Check three "
        "issues: (a) Evidence Sufficiency: are your strongest claims supported by explicit "
        "observable evidence (quest outcomes, votes, or quoted statements)? (b) Cross-Round "
        "Consistency: does your deduction stay consistent with prior rounds unless new evidence "
        "justifies a revision? (c) Public Statement Alignment: your statement field must be what "
        "you would plausibly say aloud to other players, strategic yet grounded in observable "
        "evidence. Revise suspicion levels, beliefs, and statement to eliminate these issues.",
    ]

    if is_final_round:
        lines += [
            "",
            "FINAL ROUND: This is your definitive identification. "
            "You must name exactly 2 Evil players in deduction; no more, no fewer.",
        ]

    lines += [
        "",
        f"Now produce the final JSON, maintaining speaking order ({order_str}) "
        "in abduction, suspicion, and beliefs:",
        OUTPUT_SCHEMA_REMINDER,
    ]

    round_block = "\n".join(lines)
    return STATIC_CONTEXT + "\n\n---\n\n" + round_block


# Quick prompt sanity check
sample_game = sorted(df["game_id"].unique())[0]
g = get_game_rounds(sample_game)
ctx_map = build_cumulative_history_and_discussion(g)
protagonist = get_protagonist(sample_game)
first_round = sorted(ctx_map.keys(), key=lambda x: int(str(x).replace("R", "")))[0]
first_row = g[g["round_id"].astype(str).str.replace("R", "", regex=False).str.strip() == str(first_round).replace("R", "").strip()].iloc[0]
speaking_order = _get_speaking_order(str(first_row.get("discussion_log") or ""), protagonist)

test_prompt = build_react_prompt(
    game_id=sample_game,
    round_id=first_round,
    protagonist=protagonist,
    row_context=ctx_map[first_round],
    prior_react_text="",
    speaking_order=speaking_order,
)

print("Speaking order sample:", speaking_order)
print("ReAct prompt length:", len(test_prompt), "chars |", len(test_prompt.split()), "words")
print("\nSample prompt (first 500 chars):")
print(test_prompt[:500])


Speaking order sample: ['P3', 'P2', 'P4', 'P5']
ReAct prompt length: 8246 chars | 1206 words

Sample prompt (first 500 chars):
AVALON OVERVIEW
Avalon is a social deduction game of hidden roles. This dataset uses a modified 5-player variant where the Investigator (you) has a special win condition: correctly identifying both Evil players wins the game regardless of quest outcomes.

GAME SETUP
5 players total: 3 Good, 2 Evil. Evil players know each other's identities; Good players know only their own alignment. Good wins by passing 3 quests; Evil wins by failing 3. Team sizes: R1=2, R2=3, R3=2, R4=3, R5=3.

ROUND SEQUENCE



## Section 5: Output Parsing and Validation

In [8]:
# SECTION 5: Output parsing, normalization, and validation helpers

VALID_SUSPICION_LEVELS = ["Minimal", "Mild", "Moderate", "Strong", "Extreme"]
SUSPICION_RANK = {
    "Minimal": 1,
    "Mild": 2,
    "Moderate": 3,
    "Strong": 4,
    "Extreme": 5,
}

# Map non-canonical labels to the canonical 5-level scale
_SUSPICION_REMAP = {
    "very low": "Minimal",
    "low": "Mild",
    "high": "Strong",
    "very high": "Extreme",
}


def _strip_code_fences(text: str) -> str:
    text = (text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
        text = re.sub(r"```$", "", text).strip()
    return text


def _extract_json_object(text: str) -> str:
    """Return the first balanced JSON object block from free-form text."""
    start = text.find("{")
    if start == -1:
        return ""

    depth = 0
    in_string = False
    escape = False

    for idx, ch in enumerate(text[start:], start=start):
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start : idx + 1]

    return ""


def _attempt_parse_json(raw_text: str):
    """Best-effort parser tolerant to mild JSON formatting issues."""
    candidates = []
    cleaned = _strip_code_fences(raw_text)

    if cleaned:
        candidates.append(cleaned)

    extracted = _extract_json_object(cleaned)
    if extracted and extracted not in candidates:
        candidates.append(extracted)

    if extracted:
        normalized_quotes = extracted.replace("“", '"').replace("”", '"').replace("’", "'")
        if normalized_quotes not in candidates:
            candidates.append(normalized_quotes)

    errors = []
    for candidate in candidates:
        try:
            return json.loads(candidate), None
        except Exception as e_json:
            errors.append(f"json.loads failed: {e_json}")
            try:
                py_obj = ast.literal_eval(candidate)
                if isinstance(py_obj, dict):
                    return py_obj, None
            except Exception as e_lit:
                errors.append(f"ast.literal_eval failed: {e_lit}")

    return None, " | ".join(errors) if errors else "No parse candidates found"


def _expected_players(protagonist: str):
    return [p for p in ["P1", "P2", "P3", "P4", "P5"] if p != protagonist]


def _normalize_abduction(value, players, warnings):
    """Preserve model output order; fill in missing players at end."""
    out = []
    seen = set()

    for item in (value or []):
        if not isinstance(item, dict):
            continue
        player = str(item.get("player", "")).strip()
        if player not in players or player in seen:
            continue
        seen.add(player)
        good_expl = str(item.get("good_expl", "")).strip()
        evil_expl = str(item.get("evil_expl", "")).strip()
        # Fallback: some models output "thought" instead of separate good_expl/evil_expl
        thought = str(item.get("thought", "")).strip()
        if not good_expl and thought:
            good_expl = thought
            warnings.append(f"abduction.{player}: used 'thought' as good_expl fallback")
        if not good_expl:
            good_expl = f"Insufficient direct evidence to form a Good-aligned explanation for {player}."
        if not evil_expl:
            evil_expl = f"No clear Evil-aligned observation for {player} yet."
        choice = str(item.get("choice", "")).strip().title()
        if choice not in ["Good", "Evil"]:
            choice = "Good"
            warnings.append(f"abduction.{player}.choice normalized to 'Good'")
        out.append({"player": player, "good_expl": good_expl, "evil_expl": evil_expl, "choice": choice})

    # Fill in any missing players (appended at end for graceful fallback)
    for p in players:
        if p not in seen:
            warnings.append(f"abduction.{p} missing from model output; fallback appended")
            out.append({
                "player": p,
                "good_expl": f"Insufficient direct evidence to form a Good-aligned explanation for {p}.",
                "evil_expl": f"No clear Evil-aligned observation for {p} yet.",
                "choice": "Good",
            })

    return out


def _normalize_suspicion(value, players, warnings):
    """Preserve model output order; fill in missing players at end."""
    out = []
    seen = set()

    for item in (value or []):
        if not isinstance(item, dict):
            continue
        player = str(item.get("player", "")).strip()
        if player not in players or player in seen:
            continue
        seen.add(player)
        raw_level = str(item.get("level", "")).strip()
        level = _SUSPICION_REMAP.get(raw_level.lower(), raw_level.title())
        if level not in VALID_SUSPICION_LEVELS:
            level = "Moderate"
            warnings.append(f"suspicion.{player}.level '{raw_level}' invalid; defaulted to Moderate")
        reason = str(item.get("reason", "")).strip()
        if not reason:
            reason = "Insufficient direct evidence to assign a definitive level."
            warnings.append(f"suspicion.{player}.reason missing; fallback used")
        out.append({"player": player, "level": level, "reason": reason})

    # Fill in any missing players
    for p in players:
        if p not in seen:
            warnings.append(f"suspicion.{p} missing; fallback to Moderate")
            out.append({"player": p, "level": "Moderate",
                        "reason": "Insufficient direct evidence to assign a definitive level."})

    return out


def _normalize_beliefs(value, players, suspicion_map, warnings):
    """
    Returns flat list: level-1 for all 4 players (model output order, missing filled at end),
    then level-2 for Strong/Extreme players only (model output order, missing filled at end).
    """
    l1_map = {}  # insertion order preserved (Python 3.7+)
    l2_map = {}

    if isinstance(value, list):
        for item in value:
            if not isinstance(item, dict):
                continue
            player = str(item.get("player", "")).strip()
            if player not in players:
                continue
            lvl = item.get("level")
            content = str(item.get("content", "")).strip()
            if lvl == 1 and content and player not in l1_map:
                l1_map[player] = content
            elif lvl == 2 and content and player not in l2_map:
                l2_map[player] = content
            # Legacy format: {"player": "...", "level1": "...", "level2": "..."}
            v1 = str(item.get("level1", "")).strip()
            v2 = str(item.get("level2", "")).strip()
            if v1 and player not in l1_map:
                l1_map[player] = v1
            if v2 and player not in l2_map:
                l2_map[player] = v2

    out = []

    # Level-1: model output order first, then any missing players
    seen_l1 = set()
    for p in l1_map:
        if p in players:
            out.append({"level": 1, "player": p, "content": l1_map[p]})
            seen_l1.add(p)
    for p in players:
        if p not in seen_l1:
            content = f"{p}'s behavior provides partial but ambiguous evidence about their alignment."
            warnings.append(f"beliefs level-1 for {p} missing; fallback used")
            out.append({"level": 1, "player": p, "content": content})

    # Level-2: model output order for Strong/Extreme only
    for p in l2_map:
        if suspicion_map.get(p, "Minimal") in ["Strong", "Extreme"]:
            out.append({"level": 2, "player": p, "content": l2_map[p]})
    # Fill missing level-2 for Strong/Extreme players
    for p in players:
        if suspicion_map.get(p, "Minimal") in ["Strong", "Extreme"] and p not in l2_map:
            out.append({
                "level": 2, "player": p,
                "content": f"{p} appears to be managing others' perceptions of their alignment strategically.",
            })

    return out


def _normalize_deduction(value, players, suspicion_map, warnings):
    """
    Enforce Strong/Extreme players in deduction. Allow 1-3 players (prefer 2 per prompt).
    """
    if not isinstance(value, list):
        value = []

    cleaned = []
    for x in value:
        p = str(x).strip().rstrip(",").strip()
        if p in players and p not in cleaned:
            cleaned.append(p)

    # Players with Strong or Extreme suspicion must appear in deduction
    must_include = [p for p in players if suspicion_map.get(p, "Minimal") in ["Strong", "Extreme"]]
    for p in must_include:
        if p not in cleaned:
            cleaned.append(p)
            warnings.append(f"deduction: added {p} (Strong/Extreme suspicion)")

    # Ensure at least 1 player (prompt instructs to prefer 2)
    if len(cleaned) == 0:
        ranked = sorted(
            players,
            key=lambda p: SUSPICION_RANK.get(suspicion_map.get(p, "Minimal"), 0),
            reverse=True,
        )
        cleaned = ranked[:2]
        warnings.append(f"deduction empty; inferred top-2 from suspicion: {cleaned}")

    return cleaned[:3]


def _build_human_flag_trace(protagonist: str, reason: str, detail: str, raw_response: str):
    players = _expected_players(protagonist)
    trace = {
        "abduction": [
            {
                "player": p,
                "good_expl": f"Parsing failed; manual review required for {p}.",
                "evil_expl": f"Parsing failed; manual review required for {p}.",
                "choice": "Good",
            }
            for p in players
        ],
        "suspicion": [
            {"player": p, "level": "Moderate", "reason": "Automatic fallback due to malformed model output."}
            for p in players
        ],
        "depth": 1,
        "beliefs": [
            {"level": 1, "player": p, "content": "Insufficient reliable parse — manual review required."}
            for p in players
        ],
        "statement": f"{HUMAN_FLAG_TAG}: {reason}",
        "deduction": players[:2],
        "_human_review_required": True,
        "_human_review_reason": reason,
        "_human_review_detail": detail[:500],
        "_raw_response_excerpt": (raw_response or "")[:1000],
    }
    return trace


def parse_and_validate_trace(raw_response: str, protagonist: str, is_final_round: bool = False):
    """
    Never reject a trace:
    1) Parse with best effort.
    2) Normalize schema and repair missing fields.
    3) Add human-review metadata only for critical failures.
    4) Normalization warnings are logged to stdout only — not stored in the output JSON.
    """
    parsed, parse_error = _attempt_parse_json(raw_response)

    if parsed is None:
        return _build_human_flag_trace(
            protagonist=protagonist,
            reason="parse_error",
            detail=parse_error or "Unknown parsing error",
            raw_response=raw_response,
        )

    if not isinstance(parsed, dict):
        return _build_human_flag_trace(
            protagonist=protagonist,
            reason="non_object_output",
            detail=f"Parsed type: {type(parsed).__name__}",
            raw_response=raw_response,
        )

    players = _expected_players(protagonist)
    warnings = []

    normalized = {}
    normalized["abduction"] = _normalize_abduction(parsed.get("abduction"), players, warnings)
    normalized["suspicion"] = _normalize_suspicion(parsed.get("suspicion"), players, warnings)

    suspicion_map = {s["player"]: s["level"] for s in normalized["suspicion"]}

    depth = parsed.get("depth", 1)
    if depth not in [1, 2]:
        depth = 2 if any(v in ["Strong", "Extreme"] for v in suspicion_map.values()) else 1
        warnings.append("depth normalized based on suspicion profile")
    normalized["depth"] = depth

    normalized["beliefs"] = _normalize_beliefs(parsed.get("beliefs"), players, suspicion_map, warnings)

    statement = str(parsed.get("statement", "")).strip()
    if not statement:
        statement = "Insufficient confidence for definitive claim; maintaining cautious suspicion updates."
        warnings.append("statement synthesized")
    normalized["statement"] = statement

    normalized["deduction"] = _normalize_deduction(parsed.get("deduction"), players, suspicion_map, warnings)

    if is_final_round and len(normalized["deduction"]) != 2:
        issue = (
            "Final round requires exactly 2 deduction players; "
            f"got {len(normalized['deduction'])}: {normalized['deduction']}"
        )
        if normalized.get("_human_review_required"):
            normalized["_human_review_reason"] = (
                f"{normalized.get('_human_review_reason', 'unspecified')}; "
                "final_round_deduction_cardinality"
            )
            prior_detail = normalized.get("_human_review_detail", "")
            normalized["_human_review_detail"] = f"{prior_detail} | {issue}" if prior_detail else issue
        else:
            normalized["_human_review_required"] = True
            normalized["_human_review_reason"] = "final_round_deduction_cardinality"
            normalized["_human_review_detail"] = issue

    required_fields = ["abduction", "suspicion", "beliefs", "statement", "deduction"]
    missing_required = [
        k for k in required_fields
        if k not in parsed or parsed.get(k) in (None, "", [], {})
    ]

    if missing_required:
        normalized["_human_review_required"] = True
        normalized["_human_review_reason"] = "missing_required_fields"
        normalized["_human_review_detail"] = f"Missing/empty in raw output: {missing_required}"

    if warnings:
        # Log only — do not store in output JSON
        print(f"  [PARSER] normalization: {'; '.join(warnings)}")

    return normalized



def format_trace_json(trace: dict) -> str:
    """
    Serialize a reasoning trace with top-level keys on separate lines.
    Each list-of-dicts element gets its own compact single-line representation.
    Scalar lists (e.g. deduction) and scalar values are inlined on the same key line.
    Matches the format_matrix_tactic_scale pattern from the log-gen pipeline.
    """
    if not trace:
        return "{}"
    lines = ["{"]
    items = list(trace.items())
    for i, (key, value) in enumerate(items):
        comma = "," if i < len(items) - 1 else ""
        if isinstance(value, list) and value and isinstance(value[0], dict):
            # Multi-line: each dict element on its own indented line
            lines.append(f'  "{key}": [')
            for j, elem in enumerate(value):
                elem_comma = "," if j < len(value) - 1 else ""
                lines.append(
                    f'    {json.dumps(elem, ensure_ascii=False, separators=(",", ":"))}{elem_comma}'
                )
            lines.append(f'  ]{comma}')
        else:
            # Inline: scalars and scalar lists (e.g. deduction: ["P2","P3"])
            lines.append(
                f'  "{key}": {json.dumps(value, ensure_ascii=False, separators=(",", ":"))}{comma}'
            )
    lines.append("}")
    return "\n".join(lines)


print("parse_and_validate_trace() with repair/normalization defined.")


parse_and_validate_trace() with repair/normalization defined.


## Section 6: GPT-5.4 API Call Handler

In [9]:
@backoff.on_exception(backoff.expo, Exception, max_time=MAX_BACKOFF_SECONDS)
def _call_api(system_prompt: str, user_prompt: str) -> str:
    """Raw API call with exponential backoff for transient failures."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_completion_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        response_format={"type": "json_object"},
    )
    return (response.choices[0].message.content or "").strip()


def call_gpt_with_validation(
    system_prompt: str,
    user_prompt: str,
    protagonist: str,
    framework: str,
    game_id: str,
    round_id: str,
    is_final_round: bool = False,
):
    """
    Retry API + parsing until usable output is produced.
    Never return None: returns normalized trace or human-review fallback trace.
    """
    last_result = None

    for attempt in range(1, MAX_API_RETRIES + 1):
        try:
            raw = _call_api(system_prompt, user_prompt)
            result = parse_and_validate_trace(raw, protagonist, is_final_round=is_final_round)
            last_result = result

            # If repair still marks critical gaps, retry a few times before accepting fallback.
            if result.get("_human_review_required", False) and attempt < MAX_API_RETRIES:
                print(
                    f"  [{framework.upper()}] {game_id} {round_id} — "
                    f"attempt {attempt}/{MAX_API_RETRIES}: human-review output, retrying..."
                )
                continue

            return result

        except Exception as e:
            if attempt < MAX_API_RETRIES:
                print(
                    f"  [{framework.upper()}] {game_id} {round_id} — "
                    f"attempt {attempt}/{MAX_API_RETRIES}: API error: {e}; retrying..."
                )
                time.sleep(min(2 ** attempt, 20))
            else:
                print(
                    f"  [{framework.upper()}] {game_id} {round_id} — "
                    f"all retries exhausted; writing human-review fallback."
                )

    if last_result is not None:
        return last_result

    return _build_human_flag_trace(
        protagonist=protagonist,
        reason="api_failure",
        detail=f"API failed after {MAX_API_RETRIES} attempts",
        raw_response="",
    )


print("API call handler defined (robust retries + fallback).")


API call handler defined (robust retries + fallback).


## Section 7: Sequential Round Generation (Per Game)

In [10]:
def _is_human_flag_trace(trace: dict) -> bool:
    return isinstance(trace, dict) and bool(trace.get("_human_review_required", False))


def generate_game_traces(game_id: str) -> dict:
    """
    Generate reasoning traces for all rounds of one game.
    - Keeps independent prior chains for ReAct and ReCon.
    - Uses cumulative context up to each round.
    - Never drops a round: each round gets a normalized trace or human-review fallback.
    """
    game_rounds = get_game_rounds(game_id)
    protagonist = get_protagonist(game_id)
    ctx_map = build_cumulative_history_and_discussion(game_rounds)

    react_prior = []  # list[(round_id, trace)]
    recon_prior = []
    results = {"react": {}, "recon": {}}

    # Format the system prompt with this game's protagonist (personalized role assignment)
    system_prompt = SYSTEM_PROMPT.format(protagonist_id=protagonist)

    # Sorted round IDs for final-round detection
    sorted_round_ids = sorted(game_rounds["round_id"].astype(str).tolist(), key=_round_sort_key)

    for _, row in game_rounds.iterrows():
        round_id = str(row["round_id"])
        is_final_round = (round_id == sorted_round_ids[-1])
        row_context = ctx_map.get(round_id, {})
        speaking_order = _get_speaking_order(str(row.get("discussion_log") or ""), protagonist)

        # ReAct
        react_prompt = build_react_prompt(
            game_id=game_id,
            round_id=round_id,
            protagonist=protagonist,
            row_context=row_context,
            prior_react_text=format_prior_traces(react_prior, "ReAct"),
            speaking_order=speaking_order,
            is_final_round=is_final_round,
        )
        react_result = call_gpt_with_validation(
            system_prompt=system_prompt,
            user_prompt=react_prompt,
            protagonist=protagonist,
            framework="react",
            game_id=game_id,
            round_id=round_id,
            is_final_round=is_final_round,
        )
        results["react"][round_id] = react_result
        react_prior.append((round_id, react_result))

        # ReCon
        recon_prompt = build_recon_prompt(
            game_id=game_id,
            round_id=round_id,
            protagonist=protagonist,
            row_context=row_context,
            prior_recon_text=format_prior_traces(recon_prior, "ReCon"),
            speaking_order=speaking_order,
            is_final_round=is_final_round,
        )
        recon_result = call_gpt_with_validation(
            system_prompt=system_prompt,
            user_prompt=recon_prompt,
            protagonist=protagonist,
            framework="recon",
            game_id=game_id,
            round_id=round_id,
            is_final_round=is_final_round,
        )
        results["recon"][round_id] = recon_result
        recon_prior.append((round_id, recon_result))

    total_rounds = len(game_rounds)
    react_ok = sum(1 for v in results["react"].values() if not _is_human_flag_trace(v))
    recon_ok = sum(1 for v in results["recon"].values() if not _is_human_flag_trace(v))

    print(f"  {game_id}: {total_rounds} rounds | ReAct usable {react_ok}/{total_rounds} | ReCon usable {recon_ok}/{total_rounds}")
    return results



## Section 8: Incremental Save to CSV

In [11]:
def _round_sort_key(round_id: str) -> int:
    try:
        return int(str(round_id).replace("R", ""))
    except Exception:
        return 999


def _build_framework_rows(game_id: str, game_traces: dict, framework: str, col_name: str) -> pd.DataFrame:
    game_rows = df[df["game_id"] == game_id].copy()
    game_rows["_round_num"] = game_rows["round_id"].map(_round_sort_key)
    game_rows = game_rows.sort_values("_round_num").drop(columns=["_round_num"])

    out_rows = game_rows[PRESERVE_COLS].copy()
    out_rows[col_name] = out_rows["round_id"].apply(
        lambda rid: format_trace_json(game_traces[framework].get(str(rid)) or {})
    )
    return out_rows


def _upsert_game_rows(csv_path: Path, game_id: str, out_rows: pd.DataFrame) -> None:
    if csv_path.exists():
        existing = pd.read_csv(csv_path)
        if "game_id" in existing.columns:
            existing = existing[existing["game_id"] != game_id]
        combined = pd.concat([existing, out_rows], ignore_index=True)
    else:
        combined = out_rows.copy()

    combined["_game_num"] = combined["game_id"].astype(str).str.replace("G", "", regex=False).astype(int)
    combined["_round_num"] = combined["round_id"].astype(str).str.replace("R", "", regex=False).astype(int)
    combined = combined.sort_values(["_game_num", "_round_num"]).drop(columns=["_game_num", "_round_num"])

    combined.to_csv(csv_path, index=False)


def append_game_to_csv(game_id: str, game_traces: dict) -> None:
    """Upsert one game's rows in both output CSVs (prevents duplicate/partial corruption)."""
    specs = [
        ("react", "reasoning_gold_react", REACT_CSV_PATH),
        ("recon", "reasoning_gold_recon", RECON_CSV_PATH),
    ]

    for framework, col_name, csv_path in specs:
        out_rows = _build_framework_rows(game_id, game_traces, framework, col_name)
        _upsert_game_rows(csv_path, game_id, out_rows)

    print(f"  Upserted {game_id} to CSVs.")


def flush_pending_games(pending_game_traces: dict) -> None:
    if not pending_game_traces:
        return
    for game_id in sorted(pending_game_traces.keys()):
        append_game_to_csv(game_id, pending_game_traces[game_id])
    pending_game_traces.clear()


def _load_complete_games_from_csv(csv_path: Path, col_name: str) -> set:
    if not csv_path.exists():
        return set()

    out = pd.read_csv(csv_path, usecols=["game_id", "round_id", col_name])
    expected_per_game = df.groupby("game_id")["round_id"].nunique().to_dict()

    valid_counts = (
        out[out[col_name].notna()]
        .groupby("game_id")["round_id"]
        .nunique()
        .to_dict()
    )

    return {
        g for g, c in valid_counts.items()
        if expected_per_game.get(g, -1) == c
    }


def load_completed_games() -> set:
    """Game is completed only if both ReAct and ReCon outputs are complete."""
    react_done = _load_complete_games_from_csv(REACT_CSV_PATH, "reasoning_gold_react")
    recon_done = _load_complete_games_from_csv(RECON_CSV_PATH, "reasoning_gold_recon")
    return react_done.intersection(recon_done)


print("CSV upsert + completion helpers defined.")
print(f"Already fully completed in both CSVs: {len(load_completed_games())} games")


CSV upsert + completion helpers defined.
Already fully completed in both CSVs: 1 games


## Section 9: Batch Processing Configuration

**Configuration options:**
- `GAME_FILTER`: `None` = all 250 games. Pass a list of game IDs (e.g. `["G001","G002"]`) to test on a subset.
- `SKIP_COMPLETED`: If `True`, games already present in the output CSVs are skipped (resume-safe).
- `START_FROM`: Optional game ID to start from (useful for resuming mid-run).


In [12]:
# ── Batch Configuration ────────────────────────────────────────────────────────

# Set to a list of game IDs for test runs, or None for full run.
GAME_FILTER = ["G002"]  # Set to None for all 250 games (full run), or e.g. ["G001"] for tests.

# If True, skip only games that are complete in BOTH output CSVs.
SKIP_COMPLETED: bool = True

# Optional: start processing from a specific game ID.
START_FROM: str | None = None

# Flush buffered games to disk every N processed games.
SAVE_EVERY_N_GAMES: int = 25

# ── Prepare target and run lists ───────────────────────────────────────────────
all_game_ids = sorted(df["game_id"].unique())

if GAME_FILTER is not None:
    expected_game_ids = [g for g in all_game_ids if g in set(GAME_FILTER)]
else:
    expected_game_ids = all_game_ids.copy()

if START_FROM is not None and START_FROM in expected_game_ids:
    start_idx = expected_game_ids.index(START_FROM)
    expected_game_ids = expected_game_ids[start_idx:]

if SKIP_COMPLETED:
    already_done = load_completed_games()
else:
    already_done = set()

game_ids_to_run = [g for g in expected_game_ids if g not in already_done]
expected_rows = len(df[df["game_id"].isin(expected_game_ids)])

print(f"Expected target games : {len(expected_game_ids)}")
print(f"Expected target rows  : {expected_rows}")
print(f"Already complete      : {len(already_done)}")
print(f"Games to process now  : {len(game_ids_to_run)}")
print(f"First 5 to run        : {game_ids_to_run[:5]}")
print(f"Save cadence          : every {SAVE_EVERY_N_GAMES} games")
print(f"Model: {MODEL}  |  Temperature: {TEMPERATURE}  |  Max tokens: {MAX_TOKENS}")


Expected target games : 1
Expected target rows  : 3
Already complete      : 1
Games to process now  : 1
First 5 to run        : ['G002']
Save cadence          : every 25 games
Model: gpt-5.4  |  Temperature: 0.5  |  Max tokens: 4096


## Section 10: Main Batch Processing Loop

Run this cell to start generation. Progress is saved after every game (resume-safe).  
**For a full run:** Set `GAME_FILTER = None` in Section 9 before running this cell.


In [14]:
run_start = datetime.now()
stats = {
    "success": 0,
    "partial": 0,
    "failed": 0,
    "total_react_usable": 0,
    "total_recon_usable": 0,
    "human_flag_react": 0,
    "human_flag_recon": 0,
}

pending_game_traces = {}
games_since_flush = 0

print(f"Starting batch generation at {run_start.strftime('%H:%M:%S')}")
print(f"Processing {len(game_ids_to_run)} games...\n")

for game_id in tqdm(game_ids_to_run, desc="Games", unit="game"):
    try:
        game_traces = generate_game_traces(game_id)

    except Exception as e:
        # Ensure every round still gets a fallback entry.
        print(f"  FATAL ERROR for {game_id}: {e}")
        protagonist = get_protagonist(game_id)
        rounds = [str(r) for r in get_game_rounds(game_id)["round_id"].tolist()]
        game_traces = {"react": {}, "recon": {}}

        for rid in rounds:
            game_traces["react"][rid] = _build_human_flag_trace(
                protagonist=protagonist,
                reason="game_level_exception",
                detail=str(e),
                raw_response="",
            )
            game_traces["recon"][rid] = _build_human_flag_trace(
                protagonist=protagonist,
                reason="game_level_exception",
                detail=str(e),
                raw_response="",
            )

    pending_game_traces[game_id] = game_traces
    games_since_flush += 1

    # Periodic flush
    if games_since_flush >= SAVE_EVERY_N_GAMES:
        flush_pending_games(pending_game_traces)
        games_since_flush = 0

    n_rounds = len(df[df["game_id"] == game_id])

    react_ok = sum(1 for v in game_traces["react"].values() if not _is_human_flag_trace(v))
    recon_ok = sum(1 for v in game_traces["recon"].values() if not _is_human_flag_trace(v))

    react_human = n_rounds - react_ok
    recon_human = n_rounds - recon_ok

    stats["total_react_usable"] += react_ok
    stats["total_recon_usable"] += recon_ok
    stats["human_flag_react"] += react_human
    stats["human_flag_recon"] += recon_human

    if react_ok == n_rounds and recon_ok == n_rounds:
        stats["success"] += 1
    elif react_ok == 0 and recon_ok == 0:
        stats["failed"] += 1
    else:
        stats["partial"] += 1

# Final flush for remaining buffered games
flush_pending_games(pending_game_traces)

run_end = datetime.now()
elapsed = (run_end - run_start).total_seconds()

print(f"\n{'='*60}")
print(f"Batch complete at {run_end.strftime('%H:%M:%S')}  ({elapsed:.0f}s elapsed)")
print(f"Games fully usable     : {stats['success']}")
print(f"Games partially usable : {stats['partial']}")
print(f"Games fully flagged    : {stats['failed']}")
print(f"Usable ReAct traces    : {stats['total_react_usable']}")
print(f"Usable ReCon traces    : {stats['total_recon_usable']}")
print(f"Human flags (ReAct)    : {stats['human_flag_react']}")
print(f"Human flags (ReCon)    : {stats['human_flag_recon']}")
print(f"Output: {REACT_CSV_PATH.resolve()}")
print(f"Output: {RECON_CSV_PATH.resolve()}")


Starting batch generation at 21:49:32
Processing 1 games...



Games:   0%|          | 0/1 [00:00<?, ?game/s]

  G002: 3 rounds | ReAct usable 3/3 | ReCon usable 3/3
  Upserted G002 to CSVs.

Batch complete at 21:50:57  (85s elapsed)
Games fully usable     : 1
Games partially usable : 0
Games fully flagged    : 0
Usable ReAct traces    : 3
Usable ReCon traces    : 3
Human flags (ReAct)    : 0
Human flags (ReCon)    : 0
Output: D:\Projects\Avalon-deception\Datasets\reasoning\reasoning_react.csv
Output: D:\Projects\Avalon-deception\Datasets\reasoning\reasoning_recon.csv


## Section 11: Review Generated Traces

Inspect the output CSVs after a test run. Compare a known-bad seed (G018 R1) against the newly generated trace.


In [16]:
def _round_to_int(v):
    s = str(v).strip().upper().replace("R", "")
    try:
        return int(s)
    except Exception:
        return None


def review_trace(csv_path: Path, reasoning_col: str, game_id: str, round_id) -> None:
    """Print a formatted review of one generated trace."""
    if not csv_path.exists():
        print(f"CSV not found: {csv_path}")
        return

    out = pd.read_csv(csv_path)
    target_round = _round_to_int(round_id)

    row = out[
        (out["game_id"] == game_id)
        & (out["round_id"].apply(_round_to_int) == target_round)
    ]
    if row.empty:
        print(f"No trace found for {game_id} {round_id} in {csv_path.name}")
        return

    raw_trace = row.iloc[0][reasoning_col]
    if pd.isna(raw_trace):
        print(f"{game_id} {round_id}: trace is NaN")
        return

    try:
        trace = json.loads(raw_trace)
    except Exception as e:
        print(f"Trace JSON decode error: {e}")
        print(str(raw_trace)[:500])
        return

    print(f"{'='*70}")
    print(f"{csv_path.name} — {game_id} R{target_round}")
    print(f"{'='*70}")
    print(f"Protagonist: {get_protagonist(game_id)}")
    print(f"Player roles: {row.iloc[0]['player_roles']}")
    print()

    print("ABDUCTION:")
    for a in trace.get("abduction", []):
        print(f"  {a.get('player')}: choice={a.get('choice')}")
        if a.get("good_expl") is not None or a.get("evil_expl") is not None:
            print(f"    good_expl: {a.get('good_expl')}")
            print(f"    evil_expl: {a.get('evil_expl')}")
        elif a.get("thought"):
            # Legacy compatibility for older traces
            print(f"    thought: {a.get('thought')}")

    print()
    print("SUSPICION:")
    for s in trace.get("suspicion", []):
        print(f"  {s.get('player')}: {s.get('level')} | {s.get('reason')}")

    print()
    print(f"DEPTH: {trace.get('depth')}")

    print()
    print("BELIEFS:")
    for b in trace.get("beliefs", []):
        if "level" in b and "content" in b:
            print(f"  L{b.get('level')} {b.get('player')}: {b.get('content')}")
        else:
            # Legacy compatibility for older traces
            print(f"  {b.get('player')}: level1={b.get('level1')}")
            if b.get("level2"):
                print(f"     level2={b.get('level2')}")

    print()
    print(f"STATEMENT: {trace.get('statement')}")
    print(f"DEDUCTION: {trace.get('deduction')}")

    if trace.get("_normalization_warnings"):
        print(f"\nNormalization warnings: {trace.get('_normalization_warnings')}")

    if trace.get("_human_review_required"):
        print("\nHUMAN REVIEW REQUIRED")
        print(f"Reason: {trace.get('_human_review_reason')}")
        print(f"Detail: {trace.get('_human_review_detail')}")


# Example review target
print("Reviewing G002 R1 — ReAct:")
review_trace(REACT_CSV_PATH, "reasoning_gold_react", "G002", "R1")

print("\n\nReviewing G002 R1 — ReCon:")
review_trace(RECON_CSV_PATH, "reasoning_gold_recon", "G002", "R1")


Reviewing G002 R1 — ReAct:
reasoning_react.csv — G002 R1
Protagonist: P1
Player roles: {"P1":"Good",
  "P2":"Good",
  "P3":"Good",
  "P4":"Evil",
  "P5":"Evil"}

ABDUCTION:
  P3: choice=Good
    good_expl: P3 is reacting logically to the failed 2-person quest by excluding both participants until more evidence appears.
    evil_expl: P3 is using a safe consensus read to avoid taking a riskier stance on who actually failed.
  P2: choice=Good
    good_expl: P2 wants to test a fresh 3-player team that avoids both failed quest members.
    evil_expl: P2 is opportunistically steering the next team composition while hiding behind a reasonable suggestion.
  P5: choice=Good
    good_expl: P5 is making a sincere read that P4 looked more suspicious after the failed mission.
    evil_expl: P5 is pushing blame onto the correct culprit for credibility or bussing a partner.
  P4: choice=Evil
    good_expl: P4 is a Good player wrongly accusing me under pressure after being on the failed mission.
    e

In [17]:
def _count_human_flags_in_column(series: pd.Series) -> int:
    # Serialized JSON contains true/false lower-case.
    return int(series.fillna("").astype(str).str.contains('"_human_review_required": true', regex=False).sum())


def verify_generation_completeness() -> None:
    """
    Verify completeness against configured target:
    - Test run: validates against GAME_FILTER set.
    - Full run: validates against all games.
    - If a CSV does not exist yet, it is initialized (empty, correct headers) before checking.
    """
    target_ids = expected_game_ids if "expected_game_ids" in globals() else sorted(df["game_id"].unique())
    expected_rows_local = len(df[df["game_id"].isin(target_ids)])

    print(f"Expected games: {len(target_ids)}")
    print(f"Expected rows : {expected_rows_local}")

    overall_ok = True

    for csv_path, col_name, label in [
        (REACT_CSV_PATH, "reasoning_gold_react", "ReAct"),
        (RECON_CSV_PATH, "reasoning_gold_recon", "ReCon"),
    ]:
        if not csv_path.exists():
            csv_path.parent.mkdir(parents=True, exist_ok=True)
            pd.DataFrame(columns=PRESERVE_COLS + [col_name]).to_csv(csv_path, index=False)
            print(f"\n{label}: CSV not found — initialized empty at {csv_path}")
            # Fall through: the check below will report 0 rows (generation not yet run)

        out = pd.read_csv(csv_path)
        out_scope = out[out["game_id"].isin(target_ids)].copy()

        generated_rows = len(out_scope)
        has_col = col_name in out_scope.columns
        filled_rows = int(out_scope[col_name].notna().sum()) if has_col else 0
        missing_values = int(out_scope[col_name].isna().sum()) if has_col else expected_rows_local
        games_present = set(out_scope["game_id"].unique())
        missing_games = sorted(set(target_ids) - games_present)
        human_flags = _count_human_flags_in_column(out_scope[col_name]) if has_col else 0
        duplicates = int(out_scope.duplicated(subset=["game_id", "round_id"]).sum()) if len(out_scope) > 0 else 0

        ok = (
            generated_rows == expected_rows_local
            and filled_rows == expected_rows_local
            and duplicates == 0
            and len(missing_games) == 0
        )

        overall_ok = overall_ok and ok

        print(f"\n[{label}]")
        print(f"Rows generated    : {generated_rows}/{expected_rows_local}")
        print(f"Rows non-null     : {filled_rows}/{expected_rows_local}")
        print(f"Missing values    : {missing_values}")
        print(f"Duplicate rows    : {duplicates}")
        print(f"Human-flagged rows: {human_flags}")
        print(f"Missing games     : {len(missing_games)}")
        if missing_games:
            print(f"First missing games: {missing_games[:10]}")
        print(f"Status            : {'PASS' if ok else 'FAIL'}")

    print("\n" + "=" * 60)
    print(f"OVERALL COMPLETENESS: {'PASS' if overall_ok else 'FAIL'}")


verify_generation_completeness()


Expected games: 1
Expected rows : 3

[ReAct]
Rows generated    : 3/3
Rows non-null     : 3/3
Missing values    : 0
Duplicate rows    : 0
Human-flagged rows: 0
Missing games     : 0
Status            : PASS

[ReCon]
Rows generated    : 3/3
Rows non-null     : 3/3
Missing values    : 0
Duplicate rows    : 0
Human-flagged rows: 0
Missing games     : 0
Status            : PASS

OVERALL COMPLETENESS: PASS
